### Time series datasets (TranAD) (need deepod >= 0.4.0 and torch<1.13.1（≥1.10.0）)

#### 1. Synthetic Financial Datasets from Kaggle

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
# import matplotlib.pyplot as plt
from deepod.models.time_series import TranAD


# Load the data
df = pd.read_csv('new_data/ts/PSlog.csv')

# Transactions which are detected as fraud are cancelled, so for fraud detection these columns (oldbalanceOrg, newbalanceOrig, oldbalanceDest, newbalanceDest ) must not be used
df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'], inplace=True)

df = pd.get_dummies(df, columns=['type'])

labels = df['isFraud'].values
df.drop(columns=['isFraud'], inplace=True)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df)
X_scaled = X_scaled.astype(np.float32)

X_seq, y_seq = np.array(X_scaled), np.array(labels)

# Split the data
train_size = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:train_size], X_seq[train_size:]
y_train, y_test = y_seq[:train_size], y_seq[train_size:]

d:\Software\anaconda\envs\deepod_ts\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Fit the model
model = TranAD(seq_len=1, device='cpu')
model.fit(X_train, y_train)

scores = model.decision_function(X_test)

roc_auc = roc_auc_score(y_test, scores)
pr_auc = average_precision_score(y_test, scores)
print(f"TranAD ROC-AUC: {roc_auc:.4f} PR-AUC: {pr_auc:.4f}")

Epoch 1,	 L1 = 0.15511099045926874
Epoch 2,	 L1 = 0.12882363186641174
Epoch 3,	 L1 = 0.05933443456888199
Epoch 4,	 L1 = 0.0415638480335474
Epoch 5,	 L1 = 0.027440908703614365
TranAD ROC-AUC: 0.8357 PR-AUC: 0.0226


#### 2. Created datasets

In [2]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from deepod.models.time_series import TranAD

# Create datasets
rng = np.random.default_rng(0)
T_train, T_test = 20000, 40000
D =3      # Dimension                          
fs = 1.0
t_tr = np.arange(T_train) / fs
t_te = np.arange(T_test)  / fs

# Nonlinear background noise of multiple frequencies/phases + acoustic term + nonlinear activation
base_freqs = rng.uniform(0.005, 0.05, size=D)
phases     = rng.uniform(0, 2*np.pi, size=D)
amp        = rng.uniform(0.5, 1.3, size=D)

def synth(t):
    S = []
    for d in range(D):
        f = base_freqs[d]
        x = (amp[d] * np.sin(2*np.pi*f*t + phases[d])
             + 0.5*np.sin(2*np.pi*2.7*f*t + 0.3)
             + 0.3*np.sin(2*np.pi*0.7*f*t + 1.7))
        # Nonlinear distortion (slight)
        x = np.tanh(x) + 0.05*np.sin(2*np.pi*0.013*t)
        S.append(x)
    return np.stack(S, axis=1)

X_train = synth(t_tr) + rng.normal(0, 0.05, size=(T_train, D))
X_test = synth(t_te) + rng.normal(0, 0.05, size=(T_test,  D))

# Inject outliers
labels = np.zeros(T_test, dtype=np.int32)

def inject_segment(lo, hi, func):
    lo = max(0, lo); hi = min(T_test, hi)
    X_test[lo:hi] = func(X_test[lo:hi]); labels[lo:hi] = 1

# Abnormal shape: Short-time multi-channel square wave/sawtooth replacement
for s in [800]:
    width = 200
    k = min(D, 10)  
    idx = rng.choice(D, size=k, replace=False)
    def shape_anom(seg):
        tt = np.arange(seg.shape[0])
        sq = np.sign(np.sin(2*np.pi*0.12*tt))  # square wave
        saw = 2*(tt/len(tt) - np.floor(0.5 + tt/len(tt)))
        seg[:, idx] = 0.7*seg[:, idx] + 1.2*sq[:, None] + 0.8*saw[:, None]
        return seg
    inject_segment(s, s+width, shape_anom)

# Phase hits
for s in [1400]:
    width = 100
    k = min(D, 8) 
    idx = rng.choice(D, size=k, replace=False)
    def phase_flip(seg):
        tt = np.arange(seg.shape[0])
        flip = np.sin(2*np.pi*0.2*tt + np.pi)  # Phase π
        seg[:, idx] = 0.6*seg[:, idx] + 1.1*flip[:, None]
        return seg
    inject_segment(s, s+width, phase_flip)

# Frequency discontinuity
for s in [1900]:
    width = 100
    k = min(D, 12) 
    idx = rng.choice(D, size=k, replace=False)
    def freq_warp(seg):
        tt = np.arange(seg.shape[0])
        seg[:, idx] = 0.5*seg[:, idx] + 1.0*np.sin(2*np.pi*0.45*tt)[:, None]
        return seg
    inject_segment(s, s+width, freq_warp)

print(f"[DATA] Train={X_train.shape}  Test={X_test.shape}  Anom%={labels.mean():.2%}")

# Standardlization
scaler = StandardScaler().fit(X_train)
X_tr_std = scaler.transform(X_train).astype(np.float32)
X_te_std = scaler.transform(X_test).astype(np.float32)

# Sliding window（train stride=10, test stride=1)
SEQ_LEN = 100
STRIDE_TRAIN = 10

def get_subseqs(arr, seq_len, stride):
    T, d = arr.shape
    idx = np.arange(0, T-seq_len+1, stride)
    out = np.empty((len(idx), seq_len, d), dtype=np.float32)
    for k, s in enumerate(idx):
        out[k] = arr[s:s+seq_len]
    return out, idx

Xtr_win, _       = get_subseqs(X_tr_std, SEQ_LEN, STRIDE_TRAIN)
Xte_win, starts  = get_subseqs(X_te_std, SEQ_LEN, 1)   # stride=1


# Window label (marking 1 if any anomaly is covered)
win_labels = np.zeros(len(starts), dtype=np.int32)
for i, s in enumerate(starts):
    if labels[s:s+SEQ_LEN].max() == 1:
        win_labels[i] = 1




# TranAD 

SEQ = SEQ_LEN 

# Pass parameters
model = TranAD(seq_len=SEQ, epochs=50, device="cpu")


model.fit(X_tr_std)             # Use the original 2D sequence (T×D)
score_raw = np.asarray(model.decision_function(X_te_std)).ravel().astype(float)

# Unify to window-level scores (stride=1）
expected = X_te_std.shape[0] - SEQ + 1
score = score_raw[SEQ-1:] if len(score_raw) == X_te_std.shape[0] else score_raw
assert len(score) == expected == len(win_labels)

# Direction
if roc_auc_score(win_labels, score) < roc_auc_score(win_labels, -score):
    score = -score

roc = roc_auc_score(win_labels, score)
pra = average_precision_score(win_labels, score)
print(f"[TranAD] ROC-AUC={roc:.4f}  PR-AUC={pra:.4f}")

[DATA] Train=(20000, 3)  Test=(40000, 3)  Anom%=1.00%
Epoch 1,	 L1 = 1.1887784004211426
Epoch 2,	 L1 = 1.1011563566598026
Epoch 3,	 L1 = 0.9039946279742501
Epoch 4,	 L1 = 0.7973963916301727
Epoch 5,	 L1 = 0.7459604117003354
Epoch 6,	 L1 = 0.7280613536184485
Epoch 7,	 L1 = 0.7101905020800504
Epoch 8,	 L1 = 0.7176527814431624
Epoch 9,	 L1 = 0.7146601216359572
Epoch 10,	 L1 = 0.7136294056068767
Epoch 11,	 L1 = 0.718696266412735
Epoch 12,	 L1 = 0.695843601768667
Epoch 13,	 L1 = 0.6897547922351144
Epoch 14,	 L1 = 0.6921772713010962
Epoch 15,	 L1 = 0.6952540549364957
Epoch 16,	 L1 = 0.6800604869018901
Epoch 17,	 L1 = 0.6782531575723127
Epoch 18,	 L1 = 0.6665867079388011
Epoch 19,	 L1 = 0.6773422035303983
Epoch 20,	 L1 = 0.6800611560994928
Epoch 21,	 L1 = 0.6802470738237555
Epoch 22,	 L1 = 0.6585075177929618
Epoch 23,	 L1 = 0.6692807728594
Epoch 24,	 L1 = 0.6704392731189728
Epoch 25,	 L1 = 0.6780167384581133
Epoch 26,	 L1 = 0.6550394730134443
Epoch 27,	 L1 = 0.6623177365823225
Epoch 28,	 L1 =